# AC-Zero — Dataset Generation (Kaggle)

Grows a persistent, guaranteed-solvable Andrews-Curtis dataset outward from the
trivial group with `aczero dataset grow`, until the Kaggle time budget is nearly
spent, then writes the dataset **and** a statistics summary to `/kaggle/working`
(the notebook's saved output).

**Before you run**
- Settings -> **Internet: On** (needed to `pip install` from GitHub).
- Accelerator: **None / CPU** is enough — generation is CPU graph search, no GPU.
- Kaggle sessions are capped at ~12 h. `TIME_BUDGET_HOURS` below stops the grow
  with a safety margin so the dataset is always flushed to disk before the kill.
- Grow is **resumable via the Hugging Face bucket**: the dataset is pulled at
  start and pushed back at the end, so each session continues the last. The
  dataset summary is also published to the bucket's `datasets_summaries/` folder
  (keyed by the dataset name) so it is browsable on HF without a download.
- Safeguard: the dataset is also pushed to the bucket every `HF_UPLOAD_EVERY_HOURS`
  (default 3 h) mid-run, so an unexpected crash or hard-kill loses at most one
  interval of work. Set it to 0 to keep only the final push.


## 1. Configuration

In [ ]:
import os

# --- Repository -------------------------------------------------------------
REPO_URL = "https://github.com/HkHk2Prod/AlphaAC.git"
REPO_BRANCH = "main"  # branch / tag / commit to install

# --- Dataset shape ----------------------------------------------------------
RANK = 2  # group rank (catalog has 3*rank^2 primitive moves)
TOTAL_LENGTH_CAP = 0  # relator length cap; 0 = no cap (admit neighbours of any length)
SELECT = "smallest"  # frontier order: "smallest" | "weighted-random"
SEED = 0
WORKERS = 0  # 0 = all CPU cores; 1 = single in-process worker

# --- Time budget (Kaggle hard-kills at ~12 h) -------------------------------
TIME_BUDGET_HOURS = 11.5  # stop growing after this much wall-clock
SAFETY_MARGIN_MIN = 10  # head-room reserved for the final flush + summary

# --- Growth loop ------------------------------------------------------------
TARGET_PER_CHUNK = 2000  # new groups added per grow() call
CHECKPOINT_EVERY = 1000  # grow() flushes to disk every N new groups

# --- Output / resume --------------------------------------------------------
OUTPUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
DATASET_NAME = f"train_rank{RANK}.groups.json"
RESUME_FROM = ""  # e.g. "/kaggle/input/ac-zero-dataset/train_rank2.json"

# --- Annotation pass: per-move-set distances --------------------------------
# `dataset annotate` writes a companion <name>.<moveset>.annotations.json with
# each group's distance to the origin (trivial group) and its length-descent
# distance. Training's `descent` reward (02_train REWARD_MODE = "descent") reads
# the descent distance as N from the strict-ac annotation (the env applies strict
# AC moves). List the move sets to annotate; an empty list skips the pass.
ANNOTATE_MOVESETS = ["universal", "strict-ac"]

# --- Hugging Face bucket (dataset storage; too big for GitHub) ---------------
# The dataset lives in a HF bucket instead of git. On start the notebook pulls
# the current dataset (resume); on finish it pushes the grown one back, so each
# session continues where the last left off. Add an `HF_TOKEN` Kaggle secret
# (Add-ons -> Secrets) with write access to the bucket namespace.
HF_BUCKET = "HkHk2Prod/alphaac-data"  # namespace/bucket holding the datasets
HF_REMOTE_NAME = DATASET_NAME  # object name inside the bucket
HF_DOWNLOAD_ON_START = True  # resume by pulling the current dataset from the bucket
HF_UPLOAD_ON_FINISH = True  # push the grown dataset back to the bucket when done
# Safeguard: besides the final push, also flush whatever is on disk to the bucket
# every this many hours, so a crash or Kaggle hard-kill loses at most one interval
# of work instead of the whole session. Set to 0 to disable (final push only).
HF_UPLOAD_EVERY_HOURS = 3

## 2. Install AC-Zero from GitHub

In [ ]:
import subprocess
import sys


# --no-deps keeps Kaggle's pre-built torch/numpy; we add only the light deps the
# generation path actually needs (the grow search itself is pure-Python).
def pip_install(*args):
    # Quiet install: capture pip's output and surface it only on real failure.
    # Kaggle's base image ships a broken google-adk dependency, so pip prints a
    # "dependency resolver" ERROR on every run even when our install succeeds;
    # capturing keeps that unrelated noise (and download progress bars) out of
    # the log while still raising if the install itself fails.
    proc = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--progress-bar", "off", *args],
        capture_output=True,
        text=True,
    )
    if proc.returncode:
        print(proc.stdout, proc.stderr, sep="\n")
        proc.check_returncode()


spec = f"git+{REPO_URL}@{REPO_BRANCH}"
pip_install("--no-deps", spec)
pip_install(
    "numpy>=1.26",
    "pydantic>=2",
    "pyyaml>=6",
    "typer>=0.12",
    "rich>=13",
    "gymnasium>=0.29",
    "huggingface_hub>=1.21",
)

print("Installed ac_zero from", spec)

## 3. Time budget

In [ ]:
import time

START_TIME = time.monotonic()
DEADLINE = START_TIME + TIME_BUDGET_HOURS * 3600
MARGIN_S = SAFETY_MARGIN_MIN * 60


def elapsed_s():
    return time.monotonic() - START_TIME


def time_left_s():
    return DEADLINE - time.monotonic()


def hms(sec):
    sec = max(0, int(sec))
    h, r = divmod(sec, 3600)
    m, s = divmod(r, 60)
    return f"{h:d}h{m:02d}m{s:02d}s"


class DeadlineReached(Exception):
    """Raised from a progress callback to stop cleanly before the Kaggle kill."""


# --- Periodic HF safeguard --------------------------------------------------
# The progress callbacks below call periodic_upload() on every tick; it self-gates
# on HF_UPLOAD_EVERY_HOURS and pushes whatever is currently flushed to disk (the
# last checkpoint) back to the bucket. No-op when uploads are disabled or the
# interval is 0. Timer state lives here so it persists across the grow loop.
UPLOAD_INTERVAL_S = HF_UPLOAD_EVERY_HOURS * 3600
_last_upload_t = START_TIME


def periodic_upload(items):
    """Push (local_path, remote_name) pairs to the bucket if the interval elapsed."""
    global _last_upload_t
    if not HF_UPLOAD_ON_FINISH or UPLOAD_INTERVAL_S <= 0:
        return
    if time.monotonic() - _last_upload_t < UPLOAD_INTERVAL_S:
        return
    from pathlib import Path

    from ac_zero.datasets.hub import upload_dataset

    for local_path, remote_name in items:
        p = Path(local_path)
        if not p.exists():
            continue
        try:
            upload_dataset(p, remote_name=remote_name, bucket=HF_BUCKET)
            print(f"[{hms(elapsed_s())}] periodic HF save: {p.name}")
        except Exception as exc:
            print(f"[{hms(elapsed_s())}] periodic HF save skipped for {p.name}: {exc}")
    _last_upload_t = time.monotonic()


print(
    f"Budget {TIME_BUDGET_HOURS} h with a {SAFETY_MARGIN_MIN} min safety margin "
    f"(stop growing when {hms(MARGIN_S)} remain)."
)
if HF_UPLOAD_ON_FINISH and UPLOAD_INTERVAL_S > 0:
    print(f"Periodic HF safeguard: pushing to the bucket every {HF_UPLOAD_EVERY_HOURS} h.")

## 4. Resume from the Hugging Face bucket

Pull the current dataset from the bucket so this session continues growing it.
The first ever run finds nothing and starts fresh. Needs an `HF_TOKEN` secret.

In [ ]:
import os
from pathlib import Path

from ac_zero.datasets.hub import download_dataset

# Pick up the HF token from a Kaggle secret if it is not already in the env.
if not os.environ.get("HF_TOKEN"):
    try:
        from kaggle_secrets import UserSecretsClient

        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass  # no HF_TOKEN secret attached to this kernel

if HF_UPLOAD_ON_FINISH and not os.environ.get("HF_TOKEN"):
    raise RuntimeError(
        "HF_UPLOAD_ON_FINISH is True but no HF_TOKEN is available (env var or "
        "Kaggle secret) -- growing would run for the whole time budget and then "
        "fail to publish the result. Add an HF_TOKEN Kaggle secret (Add-ons -> "
        "Secrets) with write access, or set HF_UPLOAD_ON_FINISH = False."
    )

out_path = Path(OUTPUT_DIR) / DATASET_NAME
out_path.parent.mkdir(parents=True, exist_ok=True)

if HF_DOWNLOAD_ON_START:
    try:
        pulled = download_dataset(
            out_path, remote_name=HF_REMOTE_NAME, bucket=HF_BUCKET, missing_ok=True
        )
        if pulled is None:
            print("No dataset in the bucket yet - starting a fresh graph.")
        else:
            print(
                f"Resumed {pulled} ({pulled.stat().st_size / 1e6:.1f} MB) "
                f"from hf://buckets/{HF_BUCKET}/{HF_REMOTE_NAME}"
            )
    except Exception as exc:
        print("Bucket download skipped:", exc)

## 5. Grow the dataset

In [ ]:
import shutil
from pathlib import Path

from ac_zero.datasets.grow import GrowConfig, grow_dataset

out_path = Path(OUTPUT_DIR) / DATASET_NAME
out_path.parent.mkdir(parents=True, exist_ok=True)

# Optional resume: seed the working file from a previous session's output; grow
# then continues from the accumulated frontier and only ever adds groups.
if RESUME_FROM and Path(RESUME_FROM).exists() and Path(RESUME_FROM).resolve() != out_path.resolve():
    shutil.copy(RESUME_FROM, out_path)
    print("Resuming from", RESUME_FROM)


def progress(message, metrics):
    # grow() checkpoints between rounds, so raising here loses at most the groups
    # added since the last checkpoint — the dataset on disk stays consistent.
    if time_left_s() <= MARGIN_S:
        raise DeadlineReached(message)
    # Safeguard: periodically push the last checkpoint to the bucket (self-gated).
    periodic_upload([(out_path, HF_REMOTE_NAME)])
    if message in ("checkpoint", "grow complete") or "added" in metrics:
        print(f"[{hms(elapsed_s())}] {message}: {metrics}")


total_added = 0
chunks = 0
try:
    while time_left_s() > MARGIN_S:
        cfg = GrowConfig(
            rank=RANK,
            target=TARGET_PER_CHUNK,
            select=SELECT,
            seed=SEED,
            total_length_cap=TOTAL_LENGTH_CAP,
            workers=WORKERS,
            checkpoint_every=CHECKPOINT_EVERY,
            log_every=max(1, TARGET_PER_CHUNK // 4),
        )
        report = grow_dataset(out_path, cfg, progress=progress)
        total_added += report.added
        chunks += 1
        print(
            f"[{hms(elapsed_s())}] chunk {chunks}: +{report.added} groups "
            f"(total {report.total}, frontier {report.frontier}, max length {report.max_length})"
        )
        if report.added == 0:  # frontier exhausted (uncapped: only if the graph runs dry)
            print("Frontier exhausted — nothing left to add.")
            break
except DeadlineReached as stop:
    print(f"[{hms(elapsed_s())}] time budget reached ({stop}); dataset flushed to disk.")

print(f"\nAdded ~{total_added} groups across {chunks} chunk(s) in {hms(elapsed_s())}.")
print("Dataset:", out_path, f"({out_path.stat().st_size / 1e6:.1f} MB)")

## 6. Optional — per-move-set distance annotation

Annotate the grown group dataset with `aczero dataset annotate`, writing a
companion `<name>.<moveset>.annotations.json` per move set. Each entry carries the
group's distance to the origin (the trivial group) and its length-descent
distance. The `descent` reward in `02_train` (`REWARD_MODE = "descent"`) reads the
descent distance as its minimal-moves N from the **strict-ac** annotation. Set the
move sets to run in `ANNOTATE_MOVESETS` (config cell); the pass is time-boxed like
the grow.

In [ ]:
from ac_zero.datasets.annotate import AnnotateConfig, annotate, annotation_path

annotation_paths = []
if not ANNOTATE_MOVESETS:
    print("Skipping annotation (ANNOTATE_MOVESETS is empty).")
for moveset in ANNOTATE_MOVESETS:
    # max_depth=0 -> unbounded: search until a shorter neighbour is proven, no depth cut.
    acfg = AnnotateConfig(
        moveset=moveset, max_depth=0, workers=WORKERS, checkpoint_every=CHECKPOINT_EVERY
    )
    apath = annotation_path(out_path, moveset)

    def aprogress(message, metrics, moveset=moveset, apath=apath):
        if time_left_s() <= MARGIN_S:
            raise DeadlineReached(message)
        # Safeguard: periodically push the last annotation checkpoint (self-gated).
        periodic_upload([(apath, apath.name)])
        print(f"[{hms(elapsed_s())}] {moveset} {message}: {metrics}")

    try:
        arep = annotate(out_path, acfg, progress=aprogress)
        annotation_paths.append(apath)
        print(f"annotate[{moveset}]:", arep)
    except DeadlineReached:
        annotation_paths.append(apath)
        print(f"{moveset} annotation stopped at the time budget (partial, checkpointed).")
        break

## 7. Dataset summary

Render the canonical dataset summary (the same one `aczero dataset grow` writes)
to `OUTPUT_DIR` as `<name>.summary.md`. It is published to the bucket's
`datasets_summaries/` folder in the next cell so the summary is browsable on
Hugging Face without downloading the dataset.


In [ ]:
from IPython.display import Markdown, display

from ac_zero.datasets.summary import write_dataset_summary

# One canonical summary (shared with `aczero dataset grow`): distributions of
# size and transition degree, by-source counts, AC-triviality. Written to
# OUTPUT_DIR as <name>.summary.md and published to the bucket in the next cell.
summary_path = write_dataset_summary(out_path, OUTPUT_DIR)
print(f"Wrote {summary_path.name} ({summary_path.stat().st_size / 1e3:.1f} KB) in {hms(elapsed_s())}")
display(Markdown(summary_path.read_text()))


## 8. Publish the dataset to the Hugging Face bucket

Push the grown dataset (and any annotation files) back so the next session (or
`aczero dataset download`) picks it up, plus the summary into the bucket's
`datasets_summaries/` folder. Skipped if `HF_UPLOAD_ON_FINISH = False`.


In [ ]:
if HF_UPLOAD_ON_FINISH:
    from ac_zero.datasets.hub import upload_dataset
    from ac_zero.datasets.summary import summary_remote_name

    # Group dataset and every annotation file sync by basename; the summary goes
    # into the datasets_summaries/ folder, keyed by the dataset name.
    to_upload = [(out_path, HF_REMOTE_NAME)]
    to_upload += [(p, p.name) for p in annotation_paths if p.exists()]
    to_upload.append((summary_path, summary_remote_name(summary_path)))
    for path, remote in to_upload:
        try:
            uri = upload_dataset(path, remote_name=remote, bucket=HF_BUCKET)
            print("Uploaded", path.name, "to", uri)
        except Exception as exc:
            print("Bucket upload skipped for", path.name, ":", exc)
else:
    print("HF_UPLOAD_ON_FINISH is False - dataset left in", OUTPUT_DIR)
